In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================
# MITSUI Commodity Prediction Challenge
# Notebook Setup + Data Loading + Timing Checks
# ============================================================
#
# Purpose of this cell:
# 1. Load all required Kaggle competition files
# 2. Load the offline ground-truth file, if available
# 3. Identify feature columns and target columns
# 4. Confirm the local test/evaluation date range
# 5. Verify the lag label release timing
#
# Important concept:
# The lagged test label files contain both:
#   - date_id: the date when the label is released/available
#   - label_date_id: the date the label value actually belongs to
#
# For API-style evaluation, models may only use lag labels based on release date_id.
# That means when predicting date t, we can only use lag-label rows where date_id <= t.
#
# In this competition's local lag files, we observed:
#   lag 1 labels are released with a delay of 2
#   lag 2 labels are released with a delay of 3
#   lag 3 labels are released with a delay of 4
#   lag 4 labels are released with a delay of 5
#
# Therefore, training-side lag-label features should use:
#   train_labels[target].shift(lag + 1)
#
# Final RF notebook outputs:
# 1. rf_validation_predictions.csv/parquet
#    - model trained on date_id 0–1567
#    - predictions for date_id 1568–1826
#
# 2. rf_test_predictions.csv/parquet
#    - model trained on date_id 0–1826
#    - predictions for date_id 1827–1960
# ============================================================


# ============================================================
# 1. Imports
# ============================================================

import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 2. File Paths
# ============================================================

COMPETITION_PATH = "/kaggle/input/competitions/mitsui-commodity-prediction-challenge"

# Update this if your uploaded ground-truth dataset has a different path.
# This is the path you used earlier.
GROUND_TRUTH_PATH = "/kaggle/input/datasets/stephenweiler208/ground-truth"

print("Competition path exists:", os.path.exists(COMPETITION_PATH))
print("Ground truth path exists:", os.path.exists(GROUND_TRUTH_PATH))


# ============================================================
# 3. List Competition Files
# ============================================================

print("\nFiles under competition path:")

for dirname, _, filenames in os.walk(COMPETITION_PATH):
    print(dirname)
    for filename in filenames:
        print("   ", filename)


# ============================================================
# 4. Load Core Competition Data
# ============================================================

train = pd.read_csv(f"{COMPETITION_PATH}/train.csv")
train_labels = pd.read_csv(f"{COMPETITION_PATH}/train_labels.csv")
test = pd.read_csv(f"{COMPETITION_PATH}/test.csv")
target_pairs = pd.read_csv(f"{COMPETITION_PATH}/target_pairs.csv")

print("\nLoaded data shapes:")
print("train:", train.shape)
print("train_labels:", train_labels.shape)
print("test:", test.shape)
print("target_pairs:", target_pairs.shape)

display(train.head())
display(train_labels.head())
display(test.head())
display(target_pairs.head())


# ============================================================
# 5. Load Offline Ground Truth
# ============================================================
#
# This is only used for your professor's offline evaluation.
# The official Kaggle hidden test would not use this file directly.
# ============================================================

if os.path.isdir(GROUND_TRUTH_PATH):
    csv_files = [f for f in os.listdir(GROUND_TRUTH_PATH) if f.endswith(".csv")]
    
    if len(csv_files) == 0:
        raise FileNotFoundError("No CSV file found inside the ground truth folder.")
    
    ground_truth_file = os.path.join(GROUND_TRUTH_PATH, csv_files[0])

else:
    ground_truth_file = GROUND_TRUTH_PATH

solution = pd.read_csv(ground_truth_file)

print("\nGround truth file used:")
print(ground_truth_file)

print("\nGround truth shape:", solution.shape)
display(solution.head())


# ============================================================
# 6. Identify Target Columns and Feature Columns
# ============================================================

target_cols = target_pairs["target"].tolist()

# Sanity check: target columns should also exist in train_labels
missing_targets_in_labels = [c for c in target_cols if c not in train_labels.columns]

if len(missing_targets_in_labels) > 0:
    raise ValueError(f"Some target columns are missing from train_labels: {missing_targets_in_labels[:10]}")

id_cols = ["date_id"]

feature_cols = [c for c in train.columns if c not in id_cols]

print("\nNumber of target columns:", len(target_cols))
print("First 10 target columns:", target_cols[:10])

print("\nNumber of raw feature columns:", len(feature_cols))
print("First 10 feature columns:", feature_cols[:10])


# ============================================================
# 7. Drop Extremely Sparse Features
# ============================================================
#
# We remove features with more than 50% missing values.
# This dropped the US_Stock_GOLD adjusted columns earlier.
# Tree models can handle missing values, but extremely sparse columns
# can add noise and slow training.
# ============================================================

missing_threshold = 0.50

feature_missing_rate = train[feature_cols].isna().mean()

clean_feature_cols = feature_missing_rate[
    feature_missing_rate <= missing_threshold
].index.tolist()

dropped_feature_cols = feature_missing_rate[
    feature_missing_rate > missing_threshold
].index.tolist()

print("\nOriginal feature count:", len(feature_cols))
print("Clean feature count:", len(clean_feature_cols))
print("Dropped feature count:", len(dropped_feature_cols))

print("\nMost missing features:")
display(feature_missing_rate.sort_values(ascending=False).head(15))


# ============================================================
# 8. Date Range Checks
# ============================================================

print("\nDate ranges:")
print("train date_id:", train["date_id"].min(), "to", train["date_id"].max())
print("train_labels date_id:", train_labels["date_id"].min(), "to", train_labels["date_id"].max())
print("test date_id:", test["date_id"].min(), "to", test["date_id"].max())
print("ground truth date_id:", solution["date_id"].min(), "to", solution["date_id"].max())

local_test_start = solution["date_id"].min()
local_test_end = solution["date_id"].max()

print("\nLocal professor evaluation period:")
print(local_test_start, "to", local_test_end)

print("\nNon-leaky training period should be:")
print("date_id <", local_test_start)


# ============================================================
# 9. Target Metadata
# ============================================================
#
# target_pairs tells us:
#   - target: target column name
#   - lag: which lag bucket the target belongs to
#   - pair: the asset or asset pair used to define the target
# ============================================================

target_lag_map = target_pairs.set_index("target")["lag"].to_dict()
target_pair_map = target_pairs.set_index("target")["pair"].to_dict()

print("\nTarget lag counts:")
display(target_pairs["lag"].value_counts().sort_index())

print("\nExample target definitions:")
display(target_pairs.head(10))


# ============================================================
# 10. Verify Lag Label Release Timing
# ============================================================
#
# This confirms the release-date delay in the lagged label files.
# We use this to justify train_labels[target].shift(lag + 1)
# when building training lag-label features.
# ============================================================

for lag in [1, 2, 3, 4]:
    lag_df = pd.read_csv(
        f"{COMPETITION_PATH}/lagged_test_labels/test_labels_lag_{lag}.csv"
    )
    
    lag_df["release_minus_label"] = lag_df["date_id"] - lag_df["label_date_id"]
    
    print(f"\nLag {lag} file date timing:")
    display(lag_df[["date_id", "label_date_id", "release_minus_label"]].head(10))
    
    print(f"Lag {lag} release delay summary:")
    display(lag_df["release_minus_label"].describe())


# ============================================================
# 11. Basic Sanity Checks
# ============================================================

assert len(target_cols) == 424, "Expected 424 targets."
assert "date_id" in train.columns, "train must contain date_id."
assert "date_id" in train_labels.columns, "train_labels must contain date_id."
assert "date_id" in test.columns, "test must contain date_id."
assert "date_id" in solution.columns, "ground truth must contain date_id."

print("\nSetup complete.")
print("Ready for feature engineering and model training.")

Competition path exists: True
Ground truth path exists: True

Files under competition path:
/kaggle/input/competitions/mitsui-commodity-prediction-challenge
    target_pairs.csv
    train_labels.csv
    train.csv
    test.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels
    test_labels_lag_1.csv
    test_labels_lag_4.csv
    test_labels_lag_3.csv
    test_labels_lag_2.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/kaggle_evaluation
    mitsui_inference_server.py
    mitsui_gateway.py
    __init__.py
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/kaggle_evaluation/core
    templates.py
    base_gateway.py
    relay.py
    kaggle_evaluation.proto
    __init__.py
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/kaggle_evaluation/core/generated
    kaggle_evaluation_pb2.py
    kaggle_evaluation_pb2_grpc.py
    __init__.py

Loaded data shapes:
train: (1961, 558)
train_labels: (1961, 425)
test: (13

,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,FX_GBPCAD,FX_CADCHF,FX_NZDCAD,FX_NZDCHF,FX_ZAREUR,FX_NOKGBP,FX_NOKCHF,FX_ZARCHF,FX_NOKJPY,FX_ZARGBP
0,0,2264.5,7205.0,2570.0,3349.0,NaN,NaN,NaN,NaN,NaN,...,1.699987,0.776874,0.888115,0.689954,0.066653,0.090582,0.119630,0.078135,13.822740,0.059163
1,1,2228.0,7147.0,2579.0,3327.0,NaN,NaN,NaN,NaN,NaN,...,1.695279,0.778682,0.889488,0.692628,0.067354,0.091297,0.120520,0.079066,13.888146,0.059895
2,2,2250.0,7188.5,2587.0,3362.0,4684.0,4691.0,4684.0,3363.0,3367.0,...,1.692724,0.780186,0.894004,0.697490,0.067394,0.091478,0.120809,0.079287,13.983675,0.060037
3,3,2202.5,7121.0,2540.0,3354.0,4728.0,4737.0,4729.0,3430.0,3426.0,...,1.683111,0.785329,0.889439,0.698502,0.067639,0.091558,0.121021,0.079285,14.035571,0.059983
4,4,2175.0,7125.0,2604.0,3386.0,NaN,NaN,NaN,NaN,NaN,...,1.684816,0.787264,0.891042,0.701485,0.067443,0.091266,0.121055,0.078925,14.013760,0.059503


,date_id,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,...,target_414,target_415,target_416,target_417,target_418,target_419,target_420,target_421,target_422,target_423
0,0,0.005948,-0.002851,-0.004675,-0.000639,NaN,NaN,-0.006729,0.006066,NaN,...,NaN,0.021239,-0.005595,NaN,-0.004628,0.033793,NaN,0.038234,NaN,0.027310
1,1,0.005783,-0.024118,-0.007052,-0.018955,-0.031852,-0.019452,0.003002,-0.006876,-0.002042,...,0.003377,0.021372,-0.001517,0.012846,0.010547,0.030527,-0.000764,0.025021,0.003548,0.020940
2,2,0.001048,0.023836,-0.008934,-0.022060,NaN,NaN,0.037449,0.007658,NaN,...,-0.006712,0.009308,0.001857,-0.012761,-0.002345,0.017529,-0.005394,0.004835,-0.009075,0.001706
3,3,0.001700,-0.024618,0.011943,0.004778,NaN,NaN,-0.012519,-0.016896,NaN,...,NaN,0.036880,-0.015189,NaN,0.008118,0.001079,NaN,-0.015102,NaN,-0.033010
4,4,-0.003272,0.005234,0.006856,0.013312,0.023953,0.010681,-0.011649,0.002019,0.003897,...,NaN,0.004937,NaN,-0.006673,-0.016105,-0.004885,NaN,NaN,0.009514,NaN


,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,FX_CADCHF,FX_NZDCAD,FX_NZDCHF,FX_ZAREUR,FX_NOKGBP,FX_NOKCHF,FX_ZARCHF,FX_NOKJPY,FX_ZARGBP,is_scored
0,1827,2684.5,9190.0,1967.0,2942.0,13623.0,13920.0,13618.0,4696.0,4692.0,...,0.631633,0.808485,0.510666,0.051733,0.071654,0.079797,0.048828,13.631347,0.043845,True
1,1828,2691.5,9275.0,1985.0,2963.5,13640.0,13922.0,13634.0,4613.0,4613.0,...,0.633526,0.812571,0.514785,0.051802,0.071793,0.080214,0.048912,13.743387,0.043778,True
2,1829,2646.0,9284.5,1971.0,2914.0,13634.0,13923.0,13638.0,4647.0,4632.0,...,0.632156,0.811948,0.513278,0.051902,0.071630,0.080134,0.048971,13.766241,0.043774,True
3,1830,2634.0,9223.5,1967.0,2900.0,13681.5,13962.0,13680.0,4630.0,4631.0,...,0.629661,0.815155,0.513271,0.051907,0.071972,0.080325,0.048968,13.864629,0.043876,True
4,1831,2623.5,9232.0,1949.0,2846.5,13849.5,14141.0,13844.0,4699.5,4703.0,...,0.630969,0.816284,0.515051,0.051867,0.071790,0.080475,0.049027,13.847691,0.043736,True


,target,lag,pair
0,target_0,1,US_Stock_VT_adj_close
1,target_1,1,LME_PB_Close - US_Stock_VT_adj_close
2,target_2,1,LME_CA_Close - LME_ZS_Close
3,target_3,1,LME_AH_Close - LME_ZS_Close
4,target_4,1,LME_AH_Close - JPX_Gold_Standard_Futures_Close


...


Lag 4 file date timing:


,date_id,label_date_id,release_minus_label
0,1832,1827,5
1,1833,1828,5
2,1834,1829,5
3,1835,1830,5
4,1836,1831,5
5,1837,1832,5
6,1838,1833,5
7,1839,1834,5
8,1840,1835,5
9,1841,1836,5


Lag 4 release delay summary:


count    134.0
mean       5.0
std        0.0
min        5.0
25%        5.0
50%        5.0
75%        5.0
max        5.0
Name: release_minus_label, dtype: float64


Setup complete.
Ready for feature engineering and model training.


In [3]:
# ============================================================
# Step 2: Build API-Style Lag-Label Features
# ============================================================
#
# Purpose of this cell:
# 1. Create lag-label features for training using shift(lag + 1)
# 2. Create lag-label features for test by joining lagged_test_labels on release date_id
# 3. Merge market features + lag-label features into final model matrices
# 4. Create non-leaky train/evaluation masks
#
# Why shift(lag + 1)?
# The lagged_test_labels files showed:
#   lag 1 is released with delay 2
#   lag 2 is released with delay 3
#   lag 3 is released with delay 4
#   lag 4 is released with delay 5
#
# So for training, the equivalent available label feature is:
#   train_labels[target].shift(lag + 1)
#
# For test, we use the API-style release timing directly:
#   merge lagged_test_labels by date_id, not label_date_id
#
# ============================================================


# ============================================================
# 1. Build corrected lag-label features for TRAIN
# ============================================================

lag_feature_dict = {}

for target in target_cols:
    lag = int(target_lag_map[target])
    lag_feature_dict[f"lag_label_{target}"] = train_labels[target].shift(lag + 1)

train_label_lag_features = pd.DataFrame(lag_feature_dict)
train_label_lag_features.insert(0, "date_id", train_labels["date_id"].values)

# Copy defragments the DataFrame and avoids pandas fragmentation warnings later
train_label_lag_features = train_label_lag_features.copy()

print("Train lag-label features shape:", train_label_lag_features.shape)
display(train_label_lag_features.head())


# ============================================================
# 2. Build lag-label features for TEST
# ============================================================

test_label_lag_features = pd.DataFrame({"date_id": test["date_id"].values})

for lag in [1, 2, 3, 4]:
    lag_file = f"{COMPETITION_PATH}/lagged_test_labels/test_labels_lag_{lag}.csv"
    lag_df = pd.read_csv(lag_file)

    lag_targets = target_pairs.loc[target_pairs["lag"] == lag, "target"].tolist()
    available_targets = [c for c in lag_targets if c in lag_df.columns]

    temp = lag_df[["date_id"] + available_targets].copy()
    temp = temp.rename(columns={c: f"lag_label_{c}" for c in available_targets})

    # API-style: join on release date_id
    test_label_lag_features = test_label_lag_features.merge(
        temp,
        on="date_id",
        how="left"
    )

test_label_lag_features = test_label_lag_features.copy()

print("Test lag-label features shape:", test_label_lag_features.shape)
display(test_label_lag_features.head())


# ============================================================
# 3. Merge market features + lag-label features
# ============================================================

train_features = train[["date_id"] + clean_feature_cols].merge(
    train_label_lag_features,
    on="date_id",
    how="left"
)

test_features = test[["date_id"] + clean_feature_cols].merge(
    test_label_lag_features,
    on="date_id",
    how="left"
)

train_features = train_features.replace([np.inf, -np.inf], np.nan).copy()
test_features = test_features.replace([np.inf, -np.inf], np.nan).copy()

model_feature_cols = [c for c in train_features.columns if c != "date_id"]

print("Market feature count:", len(clean_feature_cols))
print("Lag-label feature count:", len(train_label_lag_features.columns) - 1)
print("Total model feature count:", len(model_feature_cols))

print("train_features shape:", train_features.shape)
print("test_features shape:", test_features.shape)

display(train_features.head())
display(test_features.head())


# ============================================================
# 4. Non-leaky local evaluation split
# ============================================================
#
# For professor local evaluation:
# Train only before the local ground-truth period.
# Predict/evaluate on the test dates in ground truth.
# ============================================================

train_nonleaky_mask = train_features["date_id"] < local_test_start

X_train_main = train_features.loc[train_nonleaky_mask, model_feature_cols].copy()
X_test_main = test_features[model_feature_cols].copy()

print("Non-leaky training rows:", X_train_main.shape)
print("Local test rows:", X_test_main.shape)

print(
    "Training date range:",
    train_features.loc[train_nonleaky_mask, "date_id"].min(),
    "to",
    train_features.loc[train_nonleaky_mask, "date_id"].max()
)

print(
    "Test date range:",
    test["date_id"].min(),
    "to",
    test["date_id"].max()
)


# ============================================================
# 5. Internal validation split for model/ensemble selection
# ============================================================
#
# This validation period is BEFORE the professor ground-truth period.
# Use it to pick model settings or blend weights.
# Do NOT use reconstructed test labels to choose blend weights.
# ============================================================

# This mirrors your earlier 80/20-ish split but keeps validation before 1827
blend_valid_start = 1568

inner_train_mask = train_features["date_id"] < blend_valid_start
inner_valid_mask = (
    (train_features["date_id"] >= blend_valid_start) &
    (train_features["date_id"] < local_test_start)
)

X_inner_train = train_features.loc[inner_train_mask, model_feature_cols].copy()
X_inner_valid = train_features.loc[inner_valid_mask, model_feature_cols].copy()

print("\nInternal validation setup:")
print("X_inner_train:", X_inner_train.shape)
print("X_inner_valid:", X_inner_valid.shape)

print(
    "Inner train date range:",
    train_features.loc[inner_train_mask, "date_id"].min(),
    "to",
    train_features.loc[inner_train_mask, "date_id"].max()
)

print(
    "Inner validation date range:",
    train_features.loc[inner_valid_mask, "date_id"].min(),
    "to",
    train_features.loc[inner_valid_mask, "date_id"].max()
)


# ============================================================
# 6. Final sanity checks
# ============================================================

assert X_train_main.shape[1] == X_test_main.shape[1], "Train/test feature count mismatch."
assert list(X_train_main.columns) == list(X_test_main.columns), "Train/test feature columns mismatch."

assert X_inner_train.shape[1] == X_inner_valid.shape[1], "Inner train/valid feature count mismatch."
assert list(X_inner_train.columns) == list(X_inner_valid.columns), "Inner train/valid feature columns mismatch."

print("\nFeature matrix setup complete.")
print("Ready to train internal validation models.")

Train lag-label features shape: (1961, 425)


,date_id,lag_label_target_0,lag_label_target_1,lag_label_target_2,lag_label_target_3,lag_label_target_4,lag_label_target_5,lag_label_target_6,lag_label_target_7,lag_label_target_8,...,lag_label_target_414,lag_label_target_415,lag_label_target_416,lag_label_target_417,lag_label_target_418,lag_label_target_419,lag_label_target_420,lag_label_target_421,lag_label_target_422,lag_label_target_423
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.005948,-0.002851,-0.004675,-0.000639,NaN,NaN,-0.006729,0.006066,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.005783,-0.024118,-0.007052,-0.018955,-0.031852,-0.019452,0.003002,-0.006876,-0.002042,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.001048,0.023836,-0.008934,-0.022060,NaN,NaN,0.037449,0.007658,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Test lag-label features shape: (134, 425)


,date_id,lag_label_target_0,lag_label_target_1,lag_label_target_2,lag_label_target_3,lag_label_target_4,lag_label_target_5,lag_label_target_6,lag_label_target_7,lag_label_target_8,...,lag_label_target_414,lag_label_target_415,lag_label_target_416,lag_label_target_417,lag_label_target_418,lag_label_target_419,lag_label_target_420,lag_label_target_421,lag_label_target_422,lag_label_target_423
0,1827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1829,NaN,NaN,0.017868,-0.000205,-0.016391,-0.013827,0.009972,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1830,0.002560,-0.004592,-0.001776,0.000271,-0.016696,-0.020025,0.002514,0.002204,-0.011962,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1831,0.005346,-0.014539,0.019542,0.014626,-0.011631,-0.009223,-0.005199,-0.026092,-0.003865,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Market feature count: 552
Lag-label feature count: 424
Total model feature count: 976
train_features shape: (1961, 977)
test_features shape: (134, 977)


,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,lag_label_target_414,lag_label_target_415,lag_label_target_416,lag_label_target_417,lag_label_target_418,lag_label_target_419,lag_label_target_420,lag_label_target_421,lag_label_target_422,lag_label_target_423
0,0,2264.5,7205.0,2570.0,3349.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2228.0,7147.0,2579.0,3327.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2250.0,7188.5,2587.0,3362.0,4684.0,4691.0,4684.0,3363.0,3367.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2202.5,7121.0,2540.0,3354.0,4728.0,4737.0,4729.0,3430.0,3426.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2175.0,7125.0,2604.0,3386.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,lag_label_target_414,lag_label_target_415,lag_label_target_416,lag_label_target_417,lag_label_target_418,lag_label_target_419,lag_label_target_420,lag_label_target_421,lag_label_target_422,lag_label_target_423
0,1827,2684.5,9190.0,1967.0,2942.0,13623.0,13920.0,13618.0,4696.0,4692.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1828,2691.5,9275.0,1985.0,2963.5,13640.0,13922.0,13634.0,4613.0,4613.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1829,2646.0,9284.5,1971.0,2914.0,13634.0,13923.0,13638.0,4647.0,4632.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1830,2634.0,9223.5,1967.0,2900.0,13681.5,13962.0,13680.0,4630.0,4631.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1831,2623.5,9232.0,1949.0,2846.5,13849.5,14141.0,13844.0,4699.5,4703.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Non-leaky training rows: (1827, 976)
Local test rows: (134, 976)
Training date range: 0 to 1826
Test date range: 1827 to 1960

Internal validation setup:
X_inner_train: (1568, 976)
X_inner_valid: (259, 976)
Inner train date range: 0 to 1567
Inner validation date range: 1568 to 1826

Feature matrix setup complete.
Ready to train internal validation models.


In [4]:
# ============================================================
# Optional Step 3A: Tune Random Forest Parameters on Validation
# ============================================================
#
# This tests several RF configs using only:
# Train:    date_id 0–1567
# Validate: date_id 1568–1826
#
# The best config can then be used in the final RF export cell.
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# ------------------------------------------------------------
# Competition-style scoring helpers
# ------------------------------------------------------------

SOLUTION_NULL_FILLER = -999999

def rank_correlation_sharpe_ratio(merged_df: pd.DataFrame) -> float:
    prediction_cols = [col for col in merged_df.columns if col.startswith("prediction_")]
    target_cols_metric = [col for col in merged_df.columns if col.startswith("target_")]

    daily_rank_corrs = []

    for _, row in merged_df.iterrows():
        non_null_targets = [col for col in target_cols_metric if not pd.isnull(row[col])]
        matching_predictions = [
            col for col in prediction_cols
            if col.replace("prediction", "target") in non_null_targets
        ]

        if len(non_null_targets) < 3:
            daily_rank_corrs.append(np.nan)
            continue

        if row[non_null_targets].std(ddof=0) == 0 or row[matching_predictions].std(ddof=0) == 0:
            daily_rank_corrs.append(np.nan)
            continue

        corr = np.corrcoef(
            row[matching_predictions].rank(method="average"),
            row[non_null_targets].rank(method="average")
        )[0, 1]

        daily_rank_corrs.append(corr)

    daily_rank_corrs = pd.Series(daily_rank_corrs).dropna()
    std_dev = daily_rank_corrs.std(ddof=0)

    if std_dev == 0:
        return np.nan

    return float(daily_rank_corrs.mean() / std_dev)


def score(solution_df: pd.DataFrame, submission_df: pd.DataFrame, row_id_column_name: str) -> float:
    solution_df = solution_df.copy()
    submission_df = submission_df.copy()

    del solution_df[row_id_column_name]
    del submission_df[row_id_column_name]

    assert all(solution_df.columns == submission_df.columns), "Solution and submission columns do not match."

    submission_df = submission_df.rename(
        columns={col: col.replace("target_", "prediction_") for col in submission_df.columns}
    )

    solution_df = solution_df.replace(SOLUTION_NULL_FILLER, None)

    return rank_correlation_sharpe_ratio(
        pd.concat([solution_df, submission_df], axis="columns")
    )


def evaluate_submission(submission_df, solution_df, target_cols, name="model"):
    solution_eval = solution_df[["date_id"] + target_cols].copy()
    submission_eval = submission_df[["date_id"] + target_cols].copy()

    solution_eval = solution_eval.sort_values("date_id").reset_index(drop=True)
    submission_eval = submission_eval.sort_values("date_id").reset_index(drop=True)

    assert list(solution_eval["date_id"]) == list(submission_eval["date_id"]), "date_id mismatch"
    assert list(solution_eval.columns) == list(submission_eval.columns), "column mismatch"

    model_score = score(
        solution_df=solution_eval,
        submission_df=submission_eval,
        row_id_column_name="date_id"
    )

    print(f"{name} score:", model_score)
    return model_score


# ------------------------------------------------------------
# Validation solution
# ------------------------------------------------------------

valid_solution = train_labels.loc[
    inner_valid_mask,
    ["date_id"] + target_cols
].copy().sort_values("date_id").reset_index(drop=True)


# ------------------------------------------------------------
# RF configs to test
# ------------------------------------------------------------

rf_config_grid = {
    "rf_current": {
        "n_estimators": 200,
        "max_depth": 5,
        "min_samples_leaf": 20,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1,
    },
    "rf_deeper_leaf20": {
        "n_estimators": 300,
        "max_depth": 7,
        "min_samples_leaf": 20,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1,
    },
    "rf_deeper_leaf15": {
        "n_estimators": 300,
        "max_depth": 7,
        "min_samples_leaf": 15,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1,
    },
    "rf_more_features": {
        "n_estimators": 300,
        "max_depth": 6,
        "min_samples_leaf": 20,
        "max_features": 0.35,
        "random_state": 42,
        "n_jobs": -1,
    },
    "rf_more_features_leaf15": {
        "n_estimators": 300,
        "max_depth": 6,
        "min_samples_leaf": 15,
        "max_features": 0.35,
        "random_state": 42,
        "n_jobs": -1,
    },
}


# ------------------------------------------------------------
# Train/evaluate each RF config on validation
# ------------------------------------------------------------

rf_validation_scores = {}
rf_validation_predictions_by_config = {}

for config_name, params in rf_config_grid.items():
    print("\n" + "=" * 70)
    print(f"Testing config: {config_name}")
    print(params)
    print("=" * 70)

    valid_preds = pd.DataFrame(index=X_inner_valid.index)

    start_time = time.time()

    for i, target in enumerate(target_cols):
        y_train = train_labels.loc[inner_train_mask, target].values
        target_mask = ~np.isnan(y_train)

        if target_mask.sum() < 100:
            valid_preds[target] = 0.0
            continue

        model = RandomForestRegressor(**params)

        model.fit(
            X_inner_train.loc[target_mask],
            y_train[target_mask]
        )

        valid_preds[target] = model.predict(X_inner_valid)

        if (i + 1) % 50 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"{config_name}: finished {i + 1}/{len(target_cols)} targets in {elapsed:.2f} minutes")

        del model
        gc.collect()

    valid_preds.insert(
        0,
        "date_id",
        train_features.loc[inner_valid_mask, "date_id"].values
    )

    valid_preds = valid_preds[["date_id"] + target_cols].copy()

    config_score = evaluate_submission(
        valid_preds,
        valid_solution,
        target_cols,
        name=config_name
    )

    rf_validation_scores[config_name] = config_score
    rf_validation_predictions_by_config[config_name] = valid_preds

rf_validation_scores_series = pd.Series(rf_validation_scores).sort_values(ascending=False)

print("\nRF validation config scores:")
display(rf_validation_scores_series)

best_rf_config_name = rf_validation_scores_series.index[0]
best_rf_params = rf_config_grid[best_rf_config_name]

print("\nBest RF config:")
print(best_rf_config_name)
print(best_rf_params)


Testing config: rf_current
{'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}
rf_current: finished 50/424 targets in 1.74 minutes
rf_current: finished 100/424 targets in 3.43 minutes
rf_current: finished 150/424 targets in 5.13 minutes
rf_current: finished 200/424 targets in 6.87 minutes
rf_current: finished 250/424 targets in 8.58 minutes
rf_current: finished 300/424 targets in 10.29 minutes
rf_current: finished 350/424 targets in 11.99 minutes
rf_current: finished 400/424 targets in 13.70 minutes
rf_current score: 0.2058643009031479

Testing config: rf_deeper_leaf20
{'n_estimators': 300, 'max_depth': 7, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}
rf_deeper_leaf20: finished 50/424 targets in 3.19 minutes
rf_deeper_leaf20: finished 100/424 targets in 6.37 minutes
rf_deeper_leaf20: finished 150/424 targets in 9.44 minutes
rf_deeper_leaf20: finished 200/424 targets in 12.55 minutes

KeyboardInterrupt: 

In [6]:
# ============================================================
# Step 3: Train Random Forest and Export Validation/Test Predictions
# ============================================================
#
# Purpose of this cell:
# 1. Train Random Forest models for all 424 targets on the internal training period
# 2. Predict the internal validation period and save rf_validation_predictions
# 3. Retrain Random Forest models on all pre-test data
# 4. Predict the local test period and save rf_test_predictions
#
# Why two outputs?
#
# rf_validation_predictions:
#   Used by the group to choose ensemble weights fairly.
#   Train period:      date_id 0–1599
#   Prediction period: date_id 1600–1826
#
# rf_test_predictions:
#   Used by the group for the final ensemble.
#   Train period:      date_id 0–1826
#   Prediction period: date_id 1827–1960
#
# Both outputs have the same format:
#   date_id, target_0, target_1, ..., target_423
#
# ============================================================

import gc
import time
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor


# ============================================================
# 1. Random Forest parameters
# ============================================================

rf_params = {
    "n_estimators": 300,
    "max_depth": 7,
    "min_samples_leaf": 15,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1,
}


# ============================================================
# 2. Define updated validation split: 0–1599 train, 1600–1826 validation
# ============================================================

validation_start = 1600

rf_inner_train_mask = train_features["date_id"] < validation_start
rf_inner_valid_mask = (
    (train_features["date_id"] >= validation_start) &
    (train_features["date_id"] < local_test_start)
)

X_rf_inner_train = train_features.loc[rf_inner_train_mask, model_feature_cols].copy()
X_rf_inner_valid = train_features.loc[rf_inner_valid_mask, model_feature_cols].copy()

print("RF validation split:")
print(
    "Train date range:",
    train_features.loc[rf_inner_train_mask, "date_id"].min(),
    "to",
    train_features.loc[rf_inner_train_mask, "date_id"].max()
)
print(
    "Validation date range:",
    train_features.loc[rf_inner_valid_mask, "date_id"].min(),
    "to",
    train_features.loc[rf_inner_valid_mask, "date_id"].max()
)
print("X_rf_inner_train:", X_rf_inner_train.shape)
print("X_rf_inner_valid:", X_rf_inner_valid.shape)


# ============================================================
# 3. Helper function: train one RF model per target
# ============================================================

def train_rf_and_predict(
    X_train,
    y_source,
    train_mask,
    X_predict,
    predict_date_ids,
    target_cols,
    output_name="rf_predictions"
):
    """
    Trains one Random Forest model per target and predicts a target matrix.

    Returns a DataFrame with:
        date_id, target_0, target_1, ..., target_423
    """

    predictions = pd.DataFrame(index=X_predict.index)

    start_time = time.time()

    for i, target in enumerate(target_cols):
        y_train = y_source.loc[train_mask, target].values
        target_mask = ~np.isnan(y_train)

        if target_mask.sum() < 100:
            predictions[target] = 0.0
            continue

        model = RandomForestRegressor(**rf_params)

        model.fit(
            X_train.loc[target_mask],
            y_train[target_mask]
        )

        predictions[target] = model.predict(X_predict)

        if (i + 1) % 50 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"{output_name}: finished {i + 1}/{len(target_cols)} targets in {elapsed:.2f} minutes")

        del model
        gc.collect()

    predictions.insert(0, "date_id", predict_date_ids)
    predictions = predictions[["date_id"] + target_cols].copy()

    print(f"\n{output_name} complete.")
    print(f"{output_name} shape:", predictions.shape)
    print(
        f"{output_name} date range:",
        predictions["date_id"].min(),
        "to",
        predictions["date_id"].max()
    )

    display(predictions.head())

    return predictions


# ============================================================
# 4. Train on date_id 0–1599 and predict validation 1600–1826
# ============================================================

print("=" * 70)
print("Training RF validation models")
print("Train: date_id 0–1599")
print("Predict: date_id 1600–1826")
print("=" * 70)

rf_validation_predictions = train_rf_and_predict(
    X_train=X_rf_inner_train,
    y_source=train_labels,
    train_mask=rf_inner_train_mask,
    X_predict=X_rf_inner_valid,
    predict_date_ids=train_features.loc[rf_inner_valid_mask, "date_id"].values,
    target_cols=target_cols,
    output_name="RF validation predictions"
)


# ============================================================
# 5. Save validation predictions
# ============================================================

rf_validation_predictions.to_csv(
    "/kaggle/working/rf_validation_predictions.csv",
    index=False
)

rf_validation_predictions.to_parquet(
    "/kaggle/working/rf_validation_predictions.parquet",
    index=False
)

print("\nSaved validation prediction files:")
print("/kaggle/working/rf_validation_predictions.csv")
print("/kaggle/working/rf_validation_predictions.parquet")


# ============================================================
# 6. Train on date_id 0–1826 and predict test 1827–1960
# ============================================================

print("\n" + "=" * 70)
print("Training RF final test models")
print("Train: date_id 0–1826")
print("Predict: date_id 1827–1960")
print("=" * 70)

rf_test_predictions = train_rf_and_predict(
    X_train=X_train_main,
    y_source=train_labels,
    train_mask=train_nonleaky_mask,
    X_predict=X_test_main,
    predict_date_ids=test["date_id"].values,
    target_cols=target_cols,
    output_name="RF test predictions"
)


# ============================================================
# 7. Save test predictions
# ============================================================

rf_test_predictions.to_csv(
    "/kaggle/working/rf_test_predictions.csv",
    index=False
)

rf_test_predictions.to_parquet(
    "/kaggle/working/rf_test_predictions.parquet",
    index=False
)

print("\nSaved test prediction files:")
print("/kaggle/working/rf_test_predictions.csv")
print("/kaggle/working/rf_test_predictions.parquet")


# ============================================================
# 8. Final output check
# ============================================================

print("\nFiles now available in /kaggle/working:")
for file in sorted(os.listdir("/kaggle/working")):
    print(file)

print("\nRandom Forest export step complete.")

RF validation split:
Train date range: 0 to 1599
Validation date range: 1600 to 1826
X_rf_inner_train: (1600, 976)
X_rf_inner_valid: (227, 976)
Training RF validation models
Train: date_id 0–1599
Predict: date_id 1600–1826
RF validation predictions: finished 50/424 targets in 3.38 minutes
RF validation predictions: finished 100/424 targets in 6.67 minutes
RF validation predictions: finished 150/424 targets in 9.94 minutes
RF validation predictions: finished 200/424 targets in 13.23 minutes
RF validation predictions: finished 250/424 targets in 16.49 minutes
RF validation predictions: finished 300/424 targets in 19.78 minutes
RF validation predictions: finished 350/424 targets in 23.06 minutes
RF validation predictions: finished 400/424 targets in 26.35 minutes

RF validation predictions complete.
RF validation predictions shape: (227, 425)
RF validation predictions date range: 1600 to 1826


,date_id,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,...,target_414,target_415,target_416,target_417,target_418,target_419,target_420,target_421,target_422,target_423
1600,1600,0.000194,-0.000374,0.000548,0.000488,-0.001289,0.000610,-0.000194,-0.001916,-0.001180,...,0.006676,0.001470,-0.007580,-0.001256,0.001108,0.000749,0.006286,-0.000683,-0.000421,0.001080
1601,1601,-0.000220,-0.000379,-0.000141,-0.000170,-0.001586,0.000563,0.000129,-0.001100,-0.001191,...,0.006007,0.001245,-0.007064,-0.003010,0.000144,0.000069,0.005699,0.000119,-0.001335,0.003328
1602,1602,-0.000107,-0.000346,-0.000481,0.000319,-0.001959,0.000745,0.000101,-0.001310,-0.001623,...,0.005769,0.001504,-0.008064,-0.000871,-0.000548,-0.004608,0.006645,-0.006003,0.000485,0.003125
1603,1603,0.000071,-0.000629,-0.000327,0.000046,-0.001446,0.000203,-0.000708,-0.000851,-0.001620,...,0.006192,0.001782,-0.008042,-0.001458,0.001233,-0.005681,0.004950,-0.006532,0.000512,0.003045
1604,1604,0.000431,-0.001109,0.000270,0.000371,-0.000672,0.000469,-0.001360,-0.001604,-0.001010,...,0.005778,0.000712,-0.006429,-0.005389,0.001623,-0.004036,0.004452,-0.007483,-0.004052,0.000300



Saved validation prediction files:
/kaggle/working/rf_validation_predictions.csv
/kaggle/working/rf_validation_predictions.parquet

Training RF final test models
Train: date_id 0–1826
Predict: date_id 1827–1960
RF test predictions: finished 50/424 targets in 3.90 minutes
RF test predictions: finished 100/424 targets in 7.71 minutes
RF test predictions: finished 150/424 targets in 11.48 minutes
RF test predictions: finished 250/424 targets in 19.11 minutes
RF test predictions: finished 300/424 targets in 22.91 minutes
RF test predictions: finished 350/424 targets in 26.72 minutes
RF test predictions: finished 400/424 targets in 30.53 minutes

RF test predictions complete.
RF test predictions shape: (134, 425)
RF test predictions date range: 1827 to 1960


,date_id,target_0,target_1,target_2,target_3,target_4,target_5,target_6,target_7,target_8,...,target_414,target_415,target_416,target_417,target_418,target_419,target_420,target_421,target_422,target_423
0,1827,0.000820,-0.002612,0.000765,0.002294,-0.000504,-0.001553,-0.002098,-0.003816,-0.001481,...,0.009482,0.005799,-0.004439,-0.002612,0.013170,-0.003570,0.004871,0.000735,-0.000112,-0.008297
1,1828,0.000272,-0.002225,0.000823,0.002467,-0.001116,-0.002409,-0.002542,-0.003167,-0.000807,...,0.003539,0.009417,-0.002501,-0.000047,0.012021,0.000788,-0.004000,0.002835,-0.000711,-0.002692
2,1829,0.000628,-0.002223,0.001103,0.002359,-0.000560,-0.001231,-0.001691,-0.003529,-0.001445,...,0.009130,0.005713,-0.003045,-0.001671,0.010165,-0.004960,0.005080,-0.001571,0.000232,-0.005021
3,1830,0.000591,-0.002165,0.000265,0.001998,-0.000515,-0.000264,-0.001172,-0.003212,-0.001441,...,0.008938,0.004665,-0.003022,-0.001786,0.006303,-0.004505,0.008720,-0.002779,0.000507,-0.002112
4,1831,0.000427,-0.000759,-0.001042,0.001152,-0.000757,-0.000100,-0.000951,-0.001717,-0.001717,...,0.009831,0.002475,-0.003392,-0.002811,0.004925,-0.005212,0.007788,-0.003496,0.000408,-0.000802



Saved test prediction files:
/kaggle/working/rf_test_predictions.csv
/kaggle/working/rf_test_predictions.parquet

Files now available in /kaggle/working:
.virtual_documents
rf_test_predictions.csv
rf_test_predictions.parquet
rf_validation_predictions.csv
rf_validation_predictions.parquet

Random Forest export step complete.


In [7]:
# ============================================================
# Optional: Score RF validation predictions
# This scores only the validation period 1600–1826
# ============================================================

SOLUTION_NULL_FILLER = -999999

def rank_correlation_sharpe_ratio(merged_df: pd.DataFrame) -> float:
    prediction_cols = [col for col in merged_df.columns if col.startswith("prediction_")]
    target_cols_metric = [col for col in merged_df.columns if col.startswith("target_")]

    daily_rank_corrs = []

    for _, row in merged_df.iterrows():
        non_null_targets = [col for col in target_cols_metric if not pd.isnull(row[col])]

        matching_predictions = [
            col for col in prediction_cols
            if col.replace("prediction", "target") in non_null_targets
        ]

        if len(non_null_targets) < 3:
            daily_rank_corrs.append(np.nan)
            continue

        if row[non_null_targets].std(ddof=0) == 0 or row[matching_predictions].std(ddof=0) == 0:
            daily_rank_corrs.append(np.nan)
            continue

        corr = np.corrcoef(
            row[matching_predictions].rank(method="average"),
            row[non_null_targets].rank(method="average")
        )[0, 1]

        daily_rank_corrs.append(corr)

    daily_rank_corrs = pd.Series(daily_rank_corrs).dropna()
    std_dev = daily_rank_corrs.std(ddof=0)

    if std_dev == 0:
        raise ZeroDivisionError("Denominator is zero, unable to compute Sharpe ratio.")

    print("\nDaily rank correlation summary:")
    display(daily_rank_corrs.describe())
    print("Mean daily rank correlation:", daily_rank_corrs.mean())
    print("Std daily rank correlation:", std_dev)

    return float(daily_rank_corrs.mean() / std_dev)


def score(solution_df: pd.DataFrame, submission_df: pd.DataFrame, row_id_column_name: str) -> float:
    solution_df = solution_df.copy()
    submission_df = submission_df.copy()

    del solution_df[row_id_column_name]
    del submission_df[row_id_column_name]

    assert all(solution_df.columns == submission_df.columns), "Solution and submission columns do not match."

    submission_df = submission_df.rename(
        columns={col: col.replace("target_", "prediction_") for col in submission_df.columns}
    )

    solution_df = solution_df.replace(SOLUTION_NULL_FILLER, None)

    return rank_correlation_sharpe_ratio(
        pd.concat([solution_df, submission_df], axis="columns")
    )


valid_solution = train_labels.loc[
    rf_inner_valid_mask,
    ["date_id"] + target_cols
].copy()

valid_solution = valid_solution.sort_values("date_id").reset_index(drop=True)

rf_validation_eval = rf_validation_predictions.sort_values("date_id").reset_index(drop=True)

rf_validation_score = score(
    solution_df=valid_solution,
    submission_df=rf_validation_eval,
    row_id_column_name="date_id"
)

print("\nRF validation Sharpe score:")
print(rf_validation_score)


Daily rank correlation summary:


count    227.000000
mean       0.059659
std        0.256971
min       -0.718127
25%       -0.133726
50%        0.049177
75%        0.257206
max        0.723529
dtype: float64

Mean daily rank correlation: 0.059658865040414594
Std daily rank correlation: 0.2564045096386481

RF validation Sharpe score:
0.23267478846020329
